In [1]:
from i308_calib import *
from i308_utils import *
import cv2
import pickle
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))
from calib_funcs import *
from utils import get_images_path, write_pickle

%load_ext autoreload
%autoreload 2

## Buda

In [2]:
left_file_names, right_file_names = get_images_path("../datasets/budha_board/calib", "calib", print_info=True)

Found 28 left images and 28 right images
First left image: ..\datasets\budha_board\calib\calib_left_0.jpg
First right image: ..\datasets\budha_board\calib\calib_right_0.jpg


In [3]:
# checkerboard = (10, 7)
# square_size_mm = 24.2
checkerboard, square_size_mm = load_checkerboard_config("../datasets/budha_board/checkerboard.txt")
print("checkerboard =", checkerboard)
print("square size (mm) =", square_size_mm)
checkerboard_world_points_mm = board_points(checkerboard) * square_size_mm

checkerboard = (10, 7)
square size (mm) = 24.2


In [4]:
world_points, left_images_points, right_images_points, image_size = get_points_from_images(left_file_names, right_file_names, checkerboard, checkerboard_world_points_mm)

In [5]:
err, left_K, left_dist, right_K, right_dist, R, T, E, F = cv2.stereoCalibrate(
    world_points, 
    left_images_points, 
    right_images_points, 
    None, 
    None, 
    None, 
    None, 
    image_size, 
    flags=0
)

In [6]:
to_print = [

    "# Left camera Intrinsics:",
    ("left_K", left_K),
    ("left_dist", left_dist),

    "# Right camera Intrinsics:",
    ("right_K", right_K),
    ("right_dist", right_dist),

    "# Rotation:",
    ("R", R),

    "# Translation:",
    ("T", T),
    
    "# Essential Matrix:",
    ("E", E),
    
    "# Fundamental Matrix:",
    ("F", F),
        
]
print("# STEREO CALIBRATION")
for line in to_print:

    if isinstance(line, str):   
        print(line)
    else:
        var_name, np_array = line
        print(f"{var_name} = {np_print(np_array)}\n")

# STEREO CALIBRATION
# Left camera Intrinsics:
left_K = np.array([
	[   600.569,	     0.000,	   963.433],
	[     0.000,	   600.775,	   548.961],
	[     0.000,	     0.000,	     1.000]
])

left_dist = np.array([
	[  0.004782,	 -0.023202,	 -0.000496,	  0.001601,	  0.003510]
])

# Right camera Intrinsics:
right_K = np.array([
	[   600.059,	     0.000,	   960.028],
	[     0.000,	   599.822,	   535.499],
	[     0.000,	     0.000,	     1.000]
])

right_dist = np.array([
	[  0.001949,	 -0.021507,	 -0.000628,	  0.000322,	  0.003217]
])

# Rotation:
R = np.array([
	[     1.000,	    -0.000,	     0.010],
	[     0.000,	     1.000,	     0.010],
	[    -0.010,	    -0.010,	     1.000]
])

# Translation:
T = np.array([
	[-59.300782],
	[  0.584879],
	[ -0.325488]
])

# Essential Matrix:
E = np.array([
	[    -0.006,	     0.320,	     0.588],
	[    -0.917,	    -0.571,	    59.292],
	[    -0.589,	   -59.298,	    -0.577]
])

# Fundamental Matrix:
F = np.array([
	[     0.000,	    -0.000,	    -0.001],
	[     0.0

In [7]:
calibration_results = {
    'left_K': left_K,
    'left_dist': left_dist,
    'right_K': right_K,
    'right_dist': right_dist,
    'R': R,
    'T': T,
    'E': E,
    'F': F,
    'image_size': image_size,
}

In [8]:
path = '../datasets/budha_board/calibration_results/stereo_calibration.pkl'
calibration_path = write_pickle(path, calibration_results)

In [9]:
with open(calibration_path, "rb") as f:
    calibration_results = pickle.loads(f.read())

In [10]:
print(calibration_results)

{'left_K': array([[600.56924898,   0.        , 963.43300759],
       [  0.        , 600.77485791, 548.96074764],
       [  0.        ,   0.        ,   1.        ]]), 'left_dist': array([[ 0.00478172, -0.02320239, -0.00049623,  0.00160128,  0.00351006]]), 'right_K': array([[600.05860345,   0.        , 960.02799762],
       [  0.        , 599.82241329, 535.4988342 ],
       [  0.        ,   0.        ,   1.        ]]), 'right_dist': array([[ 0.00194925, -0.0215065 , -0.00062772,  0.00032165,  0.0032174 ]]), 'R': array([[ 9.99950275e-01, -1.68257650e-04,  9.97091194e-03],
       [ 7.21585747e-05,  9.99953555e-01,  9.63751851e-03],
       [-9.97207043e-03, -9.63631979e-03,  9.99903845e-01]]), 'T': array([[-59.30078197],
       [  0.5848791 ],
       [ -0.32548765]]), 'E': array([[-5.80896883e-03,  3.19836447e-01,  5.87959752e-01],
       [-9.16823035e-01, -5.71386533e-01,  5.92918345e+01],
       [-5.89129075e-01, -5.92979294e+01, -5.77344162e-01]]), 'F': array([[ 2.48293316e-08, -1.366612

In [11]:
images_path = "../datasets/budha_board/calib"
images_prefix = "calib"
checkerboard_path = "../datasets/budha_board/checkerboard.txt"
calib_pickle_path = "../datasets/budha_board/calibration_results/stereo_calibration.pkl"
print_info = True
calib_info = {
    "images_path": images_path,
    "images_prefix": images_prefix,
    "checkerboard_path": checkerboard_path,
    "calib_pickle_path": calib_pickle_path,
    "calib_done": False,
}
file_path = write_complete_calib_pickle(calib_info, print_info)

Found 28 left images and 28 right images
First left image: ..\datasets\budha_board\calib\calib_left_0.jpg
First right image: ..\datasets\budha_board\calib\calib_right_0.jpg
checkerboard = (10, 7)
square size (mm) = 24.2
processing ..\datasets\budha_board\calib\calib_left_0.jpg ..\datasets\budha_board\calib\calib_right_0.jpg
processing ..\datasets\budha_board\calib\calib_left_1.jpg ..\datasets\budha_board\calib\calib_right_1.jpg
processing ..\datasets\budha_board\calib\calib_left_2.jpg ..\datasets\budha_board\calib\calib_right_2.jpg
processing ..\datasets\budha_board\calib\calib_left_3.jpg ..\datasets\budha_board\calib\calib_right_3.jpg
processing ..\datasets\budha_board\calib\calib_left_4.jpg ..\datasets\budha_board\calib\calib_right_4.jpg
processing ..\datasets\budha_board\calib\calib_left_5.jpg ..\datasets\budha_board\calib\calib_right_5.jpg
processing ..\datasets\budha_board\calib\calib_left_6.jpg ..\datasets\budha_board\calib\calib_right_6.jpg
processing ..\datasets\budha_board\cal

## Dataset Propio